<div style="border-left:6px solid #00897B;padding:12px 18px;background:#F5F5F5">
<span style="color:#00897B;font-weight:bold">PAUL MINCHELLA · APPRENTISSAGE EN LIGNE · VERSION ÉTUDIANTE</span>
<h1 style="color:#283593">Apprendre un jour à la fois</h1>
<b>FTL, FTRL et OGD sur un flux réel de demande issu de données ouvertes</b><br>
Exercices, contrats d’implémentation et réponses rédigées · Durée indicative : 3–4 heures
</div>

Ce TP accompagne [les diapositives du cours](../Cours_Online_Learning.pdf). Nous prévoyons le nombre quotidien de
locations de Capital Bikeshare et apprenons **après**, jamais avant, la révélation de chaque résultat.
Le jeu de données est suffisamment petit pour examiner chaque algorithme plutôt que de masquer les mises à jour dans une bibliothèque.
OGD est utile ici : une représentation bornée de faible dimension, intégrant des informations sur la demande récente,
permet un apprentissage incrémental précis. Il n'est pas nécessaire qu'il surpasse toutes les méthodes concurrentes.

### Objectifs pédagogiques

À la fin de ce TP, vous devriez savoir :

1. Explorer un jeu de données ordonné dans le temps et distinguer les covariables connues des informations indisponibles.
2. Implémenter FTL, FTRL sur les pertes complètes et OGD projeté avec une même perte convexe et un même ensemble admissible.
3. Calculer le **meilleur comparateur fixe a posteriori**, y compris pour chaque préfixe entièrement observé.
4. Sélectionner les hyperparamètres en **minimisant le regret moyen de validation**, sans utiliser les résultats du test pour cette sélection.
5. Expliquer pourquoi un faible regret moyen n'implique **ni** de petites erreurs de prévision, ni la convergence, ni un bon suivi des évolutions.

### Plan du TP

| Partie | Contenu | Durée indicative | Liens avec le cours (numéros de diapositives imprimés) |
|:--|:--|--:|:--|
| **0** | Bibliothèques, configuration et reproductibilité | 10 min | Modèle, perte et optimisation communs |
| **I** | Données ouvertes, statistiques descriptives, variables causales et découpage chronologique | 40 min | Protocole : 5–8, 9, 14 |
| **II** | Géométrie, FTL, FTRL, OGD, comparateur et sélection sur la validation | 90 min | Regret : 26–29 ; algorithmes : 32–49 ; régression : 53–55 |
| **III** | Métriques finales sur le test, diagnostics, garanties et limites du regret par unité de temps | 50 min | Portée : 50–51 ; régression ridge : 56–59 ; gestion des stocks : 60–65 |

**Version étudiante.** Complétez les corps des fonctions et rédigez votre raisonnement dans les cellules Markdown
vides prévues pour les réponses. Les noms et signatures des fonctions sont imposés ; leurs docstrings complètes restent en anglais
dans les deux versions linguistiques. Un `return` seul est un marqueur à remplacer par la sortie documentée.
Les imports, la configuration fixe et le style graphique sont fournis ; aucun algorithme d'apprentissage ni résultat numérique n'est implémenté.
Des cellules Python vides servent aux appels, contrôles, graphiques et à l'enchaînement des étapes. Définissez d'abord vos fonctions,
puis appelez-les dans l'ordre ; le notebook non complété n'exécute volontairement pas les expériences.
Affectez dans ces espaces les noms partagés documentés (notamment les tableaux et découpages du flux, `FEATURE_SCALE`,
`X_cal`, `y_cal` et `theta_ref`) avant les appels qui en dépendent. Ne modifiez pas les grilles déclarées et n'utilisez pas les résultats du test
pour la sélection. Vous pouvez ajouter des cellules, mais conservez les interfaces imposées. Enregistrez vos propres sorties dans
`online_learning_bike_sharing_student_results/fr`, jamais dans le répertoire des résultats du corrigé.
Les docstrings précisent le comportement attendu, pas une succession d'étapes d'implémentation.

Le numéro de diapositive imprimé correspond au pied de page du cours, et non au numéro de page du PDF (qui contient des étapes d'affichage superposées).

**Précision importante sur la portée.** Cet exemple pédagogique a été choisi pour son adéquation au sujet, et non comme
un banc d'essai préenregistré. La procédure de sélection du notebook n'accède pas aux résultats du test. Un unique flux
historique ne peut établir ni une supériorité universelle ni une garantie en déploiement réel.


### Nom de l’étudiant et date

# Partie 0 — Bibliothèques utiles et reproductibilité

## Exercice 0 — Préparer une expérience réexécutable depuis un noyau vierge

Importez uniquement les bibliothèques nécessaires. Utilisez un répertoire de données relatif, affichez les versions et choisissez
un style de graphique unique. Pourquoi faut-il redémarrer le noyau et exécuter toutes les cellules avant de partager le notebook ?

Les imports, les constantes fixes et le style graphique ci-dessous sont fournis. Complétez
`report_environment()` et expliquez comment vérifier la reproductibilité. Si nécessaire, installez les dépendances
depuis le dossier du notebook avec `%pip install -r requirements.txt`,
puis redémarrez le noyau. Conservez ce notebook à côté de `online_learning_bike_sharing_data` pour un usage hors ligne.


### Votre réponse — Exercice 0

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import Request, urlopen
from time import perf_counter
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import brentq, minimize
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from IPython.display import display, Markdown

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path.cwd() / "online_learning_bike_sharing_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
ARCHIVE_SHA256 = "b70182d0d0508e9abbb79306ce5c0cec34869000f8220175ac83d11dbe845401"
DAY_SHA256 = "a6bcf826782d3c0fbfdcbeead17cd0884185a0dafe8ff10cd48a874ee7ba18be"

# Declared experiment choices: none is fitted using validation or test outcomes.
TARGET_SCALE = 10_000.0
B = 4.0                      # K = {theta: ||theta||_2 <= B}
N_CAL = 90                   # initial history; unscored
N_VAL = 180                  # scored for hyperparameter selection only
FTRL_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
OGD_GRID = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]

# Figure colors; line styles and labels also identify methods.
PALETTE = {
    "teal": "#00897B",
    "skyblue": "#81D4FA",
    "blue": "#42A5F5",
    "darkblue": "#283593",
    "orange": "#FB8C00",
    "purple": "#8E24AA",
}
COLORS = {"FTL": PALETTE["skyblue"], "FTRL": PALETTE["orange"], "OGD": PALETTE["purple"]}
STYLES = {"FTL": "--", "FTRL": "-.", "OGD": "-"}
plt.rcParams.update({
    "figure.figsize": (11, 4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.2,
    "axes.titleweight": "bold", "axes.labelcolor": "#333333", "lines.linewidth": 2.0,
    "axes.prop_cycle": plt.cycler(color=[PALETTE["teal"], PALETTE["blue"], PALETTE["orange"]]),
})
pd.set_option("display.max_columns", 18)
pd.set_option("display.width", 150)


def report_environment():
    """
    Report the environment and check the conditions for a clean notebook rerun.

    Returns
    -------
    report : dict
        Keys: "versions" (a pandas Series for Python, NumPy, pandas, Matplotlib
        and SciPy), "data_dir" (Path), "seed" (int) and "figure_style" (dict).
        Display the version table and the resolved data directory as side effects.

    Notes
    -----
    Use the provided imports and fixed configuration. Do not install packages
    inside this function or change experiment choices after observing test results.
    The plotting style and constants above are supplied infrastructure, not an
    exercise solution.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## Intervalles empiriques à 95 % — portée, dépendance et tableaux enregistrés

Nous utilisons un **bootstrap apparié par blocs mobiles circulaires** : des blocs de sept jours consécutifs
sont tirés avec remise, le bouclage restant limité à l'intervalle évalué disponible.
Chaque tirage rééchantillonne les mêmes triplets `(features, outcome, recorded forecasts)` pour
toutes les méthodes. Cela préserve mieux la dépendance à court terme qu'un mélange i.i.d. des jours.
Les percentiles 2,5 et 97,5 de 1 000 statistiques rééchantillonnées forment chaque intervalle empirique.
Les bandes ont une opacité de **0,1**.

**Signification de ces intervalles.** Ils décrivent la sensibilité des trajectoires de prévision enregistrées
au schéma de rééchantillonnage empirique. Les apprenants ne sont **pas réentraînés**, la sélection
n'est **pas répétée**, et le comparateur a posteriori **est réajusté à chaque tirage**
lors du calcul du regret. Il ne s'agit pas d'intervalles décrivant les performances d'une politique adaptative
nouvellement réentraînée. Une couverture fréquentiste nécessiterait des conditions de régularité stochastique
appropriées ; la dérive saisonnière et la brièveté de ce flux peuvent les invalider.
Considérez-les comme des **intervalles empiriques conditionnels** approximatifs, et non comme une couverture à 95 % démontrée.

**Ponctuels, et non simultanés.** Chaque préfixe utilise uniquement ses propres observations, avec de nouveaux
tirages de blocs reproductibles. Nous n'utilisons pas les intervalles calculés sur toute la période aux dates précoces. Les bornes
avant 28 observations valent `NaN` : quatre blocs hebdomadaires constituent l'historique minimal annoncé.
Examiner de nombreux préfixes ou candidats ne transforme pas ces bandes en une séquence de confiance
ni en une correction pour comparaisons multiples. Le chevauchement des intervalles ne constitue pas un test.
La longueur de bloc de 7 jours est une convention hebdomadaire déclarée, et non un optimum choisi à partir des données.

Les résumés statistiques et les métriques de prévision sont accompagnés d'intervalles. **Les observations brutes,
les prévisions et erreurs individuelles enregistrées, les effectifs et étendues de l'échantillon, les exemples
pédagogiques déterministes et les bornes théoriques exactes ne sont pas accompagnés d'IC d'échantillonnage.**
Les intervalles des métriques de prévision ne doivent pas être confondus avec des intervalles de prédiction. L'étendue
bootstrap du maximum de l'échantillon ne permet pas de quantifier le risque extrême non observé. L'incertitude sur les temps
de calcul est estimée séparément à partir d'exécutions indépendantes répétées, et non de jours calendaires rééchantillonnés.


In [ ]:
# Reproducible, paired circular moving-block bootstrap.
# These settings describe a sensitivity analysis, not a distribution-free guarantee.
N_BOOT = 1000
BLOCK_LENGTH = 7
MIN_CI_N = 4 * BLOCK_LENGTH
CI_LEVEL = 0.95
CI_ALPHA = 0.10
BOOT_SEED = 202611
N_TIMING_REPEATS = 15
CI_BOUNDS, CI_AXES = {}, {}
MODEL_COLORS = {**COLORS, "Frozen calibration": PALETTE["teal"],
                "Previous day": PALETTE["darkblue"], "Previous week": PALETTE["blue"]}


def block_indices(n, generator, repetitions=N_BOOT, block_length=BLOCK_LENGTH):
    """
    Draw paired circular moving-block bootstrap indices for one scored interval.

    Parameters
    ----------
    n : int
        Positive length of the available interval.
    generator : numpy.random.Generator
        Caller-owned random generator; do not reset it or use global random state.
    repetitions : int, default N_BOOT
        Positive number of bootstrap draws.
    block_length : int, default BLOCK_LENGTH
        Positive consecutive block length; use at most n observations per block.

    Returns
    -------
    indices : ndarray of int, shape (repetitions, n)
        Indices in [0, n), with consecutive order inside each circular block.
        Truncate each concatenated draw to n positions.

    Notes
    -----
    Wrapping must remain inside this interval. Reuse these indices across
    methods to keep comparisons paired; never sample future observations.
    """


    return


def resampling_weights(indices, n):
    """
    Convert bootstrap index rows into observation multiplicities.

    Parameters
    ----------
    indices : ndarray of int, shape (repetitions, sample_length)
        Bootstrap positions taking values in [0, n).
    n : int
        Positive number of original observations.

    Returns
    -------
    weights : ndarray of int, shape (repetitions, n)
        Nonnegative occurrence counts; every row sums to sample_length.
    """


    return


def percentile_ci(samples, axis=0):
    """
    Compute a pointwise empirical percentile interval at CI_LEVEL.

    Parameters
    ----------
    samples : array_like
        Finite bootstrap statistics, possibly with several method/metric axes.
    axis : int, default 0
        Axis indexing independent bootstrap draws.

    Returns
    -------
    bounds : ndarray
        Shape of samples with axis removed, followed by a final dimension of 2.
        The last dimension stores [lower_95, upper_95] in that order.

    Notes
    -----
    These conditional empirical bounds are neither prediction intervals
    nor simultaneous or distribution-free confidence guarantees.
    """


    return


def save_ci(key, bounds, labels=None):
    """
    Validate and register interval bounds together with their axis labels.

    Parameters
    ----------
    key : str
        Unique descriptive registry key.
    bounds : array_like, shape (2,) or (n_points, 2)
        Lower/upper bounds; both endpoints must be NaN together when unavailable.
    labels : iterable or None, default None
        One label per point; use 1-based point indices when omitted.

    Returns
    -------
    registered : ndarray, shape (n_points, 2)
        Two-dimensional float bounds. Save an independent copy in CI_BOUNDS[key]
        and aligned string labels in CI_AXES[key].

    Notes
    -----
    Validate endpoint ordering for finite rows and label count. Do not fill
    early missing intervals with values computed from later observations.
    """


    return


def band(ax, x, bounds, color, label=None):
    """
    Draw a precomputed empirical interval band without changing its bounds.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Destination axes.
    x : array_like, shape (n_points,)
        Numeric or date coordinates aligned with bounds.
    bounds : ndarray, shape (n_points, 2)
        Saved lower/upper endpoints, including any leading NaN rows.
    color : str
        Matplotlib-compatible band color.
    label : str or None, default None
        Optional legend label.

    Returns
    -------
    artist : matplotlib.collections.PolyCollection
        Filled band artist with opacity CI_ALPHA and no fabricated early values.
    """


    return


def rolling_bootstrap(values, window=30, seed=0):
    """
    Compute past-only rolling means and paired block-bootstrap intervals.

    Parameters
    ----------
    values : array_like, shape (n,) or (n, n_series)
        Recorded quantities; square errors before calling if estimating MSE.
    window : int, default 30
        Window size, at least MIN_CI_N; require a full past/current window.
    seed : int, default 0
        Reproducible stream identifier combined with BOOT_SEED and endpoint.

    Returns
    -------
    estimates : ndarray, shape (n, n_series)
        Window means; NaN before the first complete window.
    intervals : ndarray, shape (n, n_series, 2)
        Pointwise paired-bootstrap bounds aligned with estimates.

    Notes
    -----
    Resample independently inside each available trailing window and pair
    columns. This function returns means, not RMSE: the caller applies the
    appropriate transformation to MSE estimates and endpoints.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

# Partie I — Statistiques descriptives et flux de données causal

## I.1 — Jeu de données ouvert et provenance

**Jeu de données :** [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset),
DOI [10.24432/C5W894](https://doi.org/10.24432/C5W894), créé par Hadi Fanaee-T.
Nous utilisons **day.csv** : **731 observations quotidiennes**, du 1er janvier 2011 au 31 décembre 2012,
Capital Bikeshare, Washington, D.C., États-Unis. Le fichier horaire n'est pas utilisé.

La page UCI du jeu de données le publie sous la licence
[Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).
Citez le jeu de données et l'article associé :

> Hadi Fanaee-T et João Gama (2013). *Étiquetage d'événements combinant des détecteurs en ensemble et
> des connaissances de contexte*. Progress in Artificial Intelligence. DOI :
> [10.1007/s13748-013-0040-3](https://doi.org/10.1007/s13748-013-0040-3).

Le fichier original `Readme.txt` est fourni sans modification. Il est antérieur à la mention de licence actuelle d'UCI
et demande de citer l'article. L'archive et le CSV quotidien sont vérifiés à l'aide d'empreintes SHA-256
fixées à l'avance, afin qu'un téléchargement modifié ne soit pas accepté silencieusement. Nous examinons individuellement les fichiers de l'archive ;
nous n'extrayons pas de chemins arbitraires depuis celle-ci.

## Exercice 1 — Examiner les observations brutes

Chargez les données quotidiennes. Vérifiez les dates, les doublons, les valeurs manquantes, les unités et l'égalité
`cnt = casual + registered`. Quelles colonnes révéleraient la réponse lors d'une prévision ?

Complétez le chargeur documenté et l'audit d'intégrité. Construisez votre tableau de disponibilité pour les informations
calendaires, les comptages passés, la météo du jour, la météo passée, `casual`, `registered` et `instant`.
Expliquez ce que mesurent les comptages de locations et leurs limites. Privilégiez les fichiers locaux vérifiés disponibles.


### Votre réponse — Exercice 1

In [ ]:
def sha256(data):
    """
    Return the SHA-256 hexadecimal digest of a byte payload.

    Parameters
    ----------
    data : bytes
        Exact file or archive bytes to verify.

    Returns
    -------
    digest : str
        Lowercase 64-character hexadecimal SHA-256 digest.
    """


    return


def load_daily_data(data_dir=DATA_DIR):
    """
    Load and validate the original UCI daily Bike Sharing data.

    Parameters
    ----------
    data_dir : pathlib.Path, default DATA_DIR
        Local cache containing day.csv and, optionally, bike_sharing.zip.
        Prefer the local verified CSV; download from DATA_URL only if needed.

    Returns
    -------
    frame : pandas.DataFrame, shape (731, 15)
        Original data indexed by parsed dteday, sorted chronologically. Preserve
        original column names and recorded weather units; dteday is not a column.

    Raises
    ------
    RuntimeError
        A needed download fails; explain how to provide the verified local CSV.
    ValueError
        A checksum differs from its declared value or a data integrity check fails.

    Notes
    -----
    Verify CSV bytes against DAY_SHA256 before parsing. If downloading or
    using a cached archive, verify ARCHIVE_SHA256 and read only the known members
    "day.csv" and "Readme.txt" without arbitrary path extraction. Use a finite
    60-second timeout for remote reads. Cache verified bytes and preserve the
    original README. Check row count, unique consecutive dates, missingness,
    and the count decomposition required by Exercise 1.
    """


    return


def audit_daily_data(daily):
    """
    Summarize data integrity, date coverage and observed units.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw daily data returned by load_daily_data, with a DatetimeIndex.

    Returns
    -------
    audit : pandas.Series
        Row/column counts, first/last dates, duplicate and missing counts,
        minimum/maximum observed rentals, and count-decomposition status.
        Display the audit and the first few rows without fitting a predictor.

    Notes
    -----
    This structural audit is not test-set model selection. Reserve outcome
    relationships and descriptive modelling for the calibration period.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## I.2 — Définir la chronologie avant d'explorer les relations

Les sept premiers jours bruts fournissent l'historique nécessaire aux retards. Parmi les **724 jours utilisables**, utilisez :

- **90 jours de calibration :** initialisation et historique non évalués ; conservés dans les objectifs
  cumulatifs de FTL/FTRL, utilisés par OGD à travers les coefficients initiaux ;
- **180 jours de validation :** choix des hyperparamètres à l'aide du regret moyen ;
- **454 jours de test :** évaluation préquentielle finale, avec des choix figés.

Seules les dates définissent ce découpage. Un mélange aléatoire entre apprentissage et test n'est pas adapté à la question
visée de prévision du futur. L'analyse descriptive des résultats est restreinte aux données de calibration
avant la sélection. Le graphique couvrant toute la période est réservé à la **partie III**.

## Exercice 2 — Construire des variables fondées strictement sur le passé

Construisez des retards de demande d'un et de sept jours, ainsi qu'une moyenne mobile sur les sept jours précédents excluant le jour courant.
Ajoutez la météo retardée et des cycles calendaires. Évitez une mise à l'échelle ajustée sur toutes les données. Comment garantir
$\|x_t\|_2\le1$ sans regarder dans le futur ?

Conservez l'échelle déclarée de demande `TARGET_SCALE = 10_000.0` ; ce n'est ni un maximum estimé ni une capacité.
Utilisez les unités météo enregistrées et un cycle annuel fixe de 365,25 jours, en conservant le 29 février.
Justifiez le diviseur commun des variables et les unités de la constante. Expliquez les conséquences d'un résultat futur dépassant
la borne déclarée sur les hypothèses, sans le tronquer silencieusement.
Dans la cellule d'exécution, affectez les sorties documentées de `prepare_stream` aux noms utilisés ensuite.
Implémentez et exécutez un contrôle indépendant de causalité des variables.


### Votre réponse — Exercice 2

In [ ]:
def make_features(frame):
    """
    Construct chronological predictors available before each outcome.

    Parameters
    ----------
    frame : pandas.DataFrame
        Raw UCI daily observations indexed by date, including the past history
        needed for lags. Keep the original normalized weather units.

    Returns
    -------
    features : pandas.DataFrame, shape (724, 12) for the full supplied data
        Date-indexed predictors, in this order: count_lag_1, count_lag_7,
        count_mean_past_7, temp_lag_1, hum_lag_1, windspeed_lag_1, annual_sin,
        annual_cos, weekly_sin, weekly_cos, workingday, holiday.
        Drop only rows without the required initial history. Do not add the
        intercept or apply the common vector divisor inside this function.

    Notes
    -----
    Scale demand-derived columns by TARGET_SCALE. Use a fixed 365.25-day
    annual cycle with day_of_year - 1 and a seven-day weekly cycle with Monday=0.
    All predictors must be reconstructible before that day's count is revealed.
    Do not use fitted full-data normalization or realized same-day weather.
    """


    return


def prepare_stream(daily, features):
    """
    Assemble the bounded linear representation and chronological split.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw date-indexed observations.
    features : pandas.DataFrame
        The unscaled predictor table returned by make_features.

    Returns
    -------
    stream : dict
        Required keys: dates (DatetimeIndex), feature_names (list beginning with
        "intercept"), d (int), FEATURE_SCALE (fixed scalar), X (n, d), y (n,),
        cal_slice, val_slice, test_slice (slice objects), N_TEST (int), and
        split_table (DataFrame of dates and sizes). For this dataset n=724.
        X includes an intercept and uses the common fixed divisor justified in
        your written answer; y is the count divided by TARGET_SCALE.

    Notes
    -----
    Use N_CAL and N_VAL, then reserve the remaining observations for test.
    Verify finite values and the stated norm/outcome bounds without clipping
    inconvenient observations. Display the chronological split table. In the
    execution workspace, bind the documented names for later exercises; no
    fitted transformation may use validation or test outcomes.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

In [ ]:
def check_feature_causality(daily, features, probe_date):
    """
    Test that a feature row uses only information available before its day.

    Parameters
    ----------
    daily : pandas.DataFrame
        Original raw observations.
    features : pandas.DataFrame
        Predictor table produced by make_features.
    probe_date : pandas.Timestamp
        A usable date with at least seven preceding daily observations.

    Returns
    -------
    None
        Raise AssertionError if a causality or trailing-window check fails.

    Notes
    -----
    Perturb today's and future counts in a copy, rebuild the predictors and
    compare the probe-date row. Independently verify its trailing mean from
    raw history. Do not mutate daily or features.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## I.3 — Statistiques descriptives à partir de l'historique initial uniquement

## Exercice 3 — Décrire avant de modéliser

Sur la période de calibration, présentez la tendance centrale, la dispersion, les valeurs manquantes et certaines corrélations.
Tracez la trajectoire de la demande, sa distribution, les différences selon le jour de la semaine et sa relation avec
la demande de la veille. Que suggèrent ces graphiques, et que ne permettent-ils **pas** d'établir ?

Utilisez uniquement les résultats de calibration. Associez les intervalles empiriques déclarés aux résumés inférentiels,
pas aux faits exacts de l'échantillon. Discutez la représentativité de la saison initiale, les facteurs de confusion et les limites
d'interprétation d'une association avec une variable retardée. Ne réglez pas les variables après examen des performances de test.


### Votre réponse — Exercice 3

In [ ]:
def describe_calibration(daily, features, y, dates, cal_slice):
    """
    Produce calibration-only summaries, associations and descriptive plots.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw date-indexed observations.
    features : pandas.DataFrame
        Chronological predictor table.
    y : ndarray, shape (n,)
        Normalized outcomes aligned with features.
    dates : pandas.DatetimeIndex
        Dates aligned with y.
    cal_slice : slice
        Initial unscored period; use no later outcomes in this analysis.

    Returns
    -------
    analysis : dict
        Include calibration (DataFrame), summary (DataFrame), summary_ci
        (DataFrame), weekday_stats (DataFrame), correlation_table (DataFrame),
        and figures (list of Matplotlib figures). Display the requested tables
        and the trajectory/distribution/weekday/previous-demand plots.

    Notes
    -----
    Report central tendency, spread and missingness for cnt, casual,
    registered, temp, hum and windspeed. Add paired block-bootstrap intervals
    for inferential summaries and save all plotted interval arrays before
    plotting. Raw observations and exact sample facts do not need intervals.
    Do not infer causality or future stationarity from descriptive association.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

# Partie II — Algorithmes en ligne et minimisation du regret moyen de validation

## II.1 — Une perte, une géométrie et un protocole communs

Pour chaque modèle, prévoyez les locations normalisées avec
$$
\widehat y_t=x_t^\top\theta_t,\qquad
\ell_t(\theta)=\tfrac12(x_t^\top\theta-y_t)^2,\qquad
K=\{\theta\in\mathbb R^d:\|\theta\|_2\le B\},\quad B=4.
$$
La même perte non régularisée et le même ensemble admissible définissent **tous** les regrets présentés.
Le régularisateur de FTRL sert à choisir les décisions ; il n'est **pas** ajouté à la perte évaluée.

```text
Au début d'une journée évaluée :
    construire x_today à partir du calendrier et des jours déjà observés
    prévoir avec theta_today ; enregistrer ce paramètre et cette prévision
    révéler y_today et évaluer 0.5 * (prediction - y_today)^2
    mettre à jour les coefficients à partir de ce résultat nouvellement révélé
    passer au jour suivant
```

## Exercice 4 — Déterminer le gradient, la courbure et la projection

Calculez $\nabla\ell_t$, $\nabla^2\ell_t$ et $\Pi_K(z)$.
La perte est-elle fortement convexe dans les $d$ directions de l'espace des paramètres ?

Incluez le vecteur nul et les cas de faisabilité dans vos contrôles de projection, et vérifiez le gradient par différences finies.
Expliquez si des données cumulées de rang plein impliquent une forte convexité pour chaque observation.
Pour un calcul à la main, prenez $K=[0,1]$, $\theta_1=0.5$, $\eta=1/4$ et les résultats 0,8 puis 0,2.
Présentez les deux étapes prévoir–révéler–mettre à jour dans un tableau, sans lire le résultat suivant prématurément.


### Votre réponse — Exercice 4

In [ ]:
def squared_loss(theta, x, target):
    """
    Evaluate the common unregularized prediction loss for one observation.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Coefficient vector.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized observed outcome.

    Returns
    -------
    loss : float
        Half squared normalized prediction error, with no FTRL penalty.
    """


    return


def squared_gradient(theta, x, target):
    """
    Evaluate the parameter gradient of squared_loss.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Current coefficient vector.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized observed outcome.

    Returns
    -------
    gradient : ndarray, shape (d,)
        Gradient in the same coordinate system as theta; do not add a penalty.
    """


    return


def project_ball(vector, radius=B):
    """
    Project a vector onto a closed Euclidean ball centered at zero.

    Parameters
    ----------
    vector : array_like, shape (d,)
        Candidate coefficient vector.
    radius : float, default B
        Positive radius of the feasible ball.

    Returns
    -------
    projected : ndarray, shape (d,)
        Closest feasible vector in Euclidean distance. Handle the zero vector
        and already feasible inputs without changing the caller's array.
    """


    return


def check_geometry(X, seed=SEED):
    """
    Validate your loss gradient and projection independently.

    Parameters
    ----------
    X : ndarray, shape (n, d)
        Feature matrix containing at least one nonzero row.
    seed : int, default SEED
        Seed for reproducible numerical examples.

    Returns
    -------
    None
        Raise AssertionError if a finite-difference gradient check, Hessian-rank
        check, zero-vector case or projected-norm check fails.

    Notes
    -----
    Design the numerical checks yourself; do not hard-code computed answers.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## II.2 — Résoudre correctement les moindres carrés sous contrainte

## Exercice 5 — Trouver un meneur sur la boule euclidienne

Soient $A=\sum x_sx_s^\top$ et $b=\sum x_sy_s$. La partie de l'objectif dépendant des pertes est
$\tfrac12\theta^\top A\theta-b^\top\theta$.
Comment la minimiser sur $K$, y compris lorsque la matrice des variables est de rang déficient ?

Énoncez le système KKT, votre règle de sélection en cas de solutions multiples et vos tolérances numériques. Traitez les contraintes
actives et inactives, les matrices de rang déficient et la matrice nulle. Vérifiez votre solveur avec un optimiseur indépendant sous contrainte.
Construisez un exemple permettant de déterminer si la projection de l'optimum non contraint suffit.


### Votre réponse — Exercice 5

In [ ]:
def solve_ball_quadratic(A, b, radius=B):
    """
    Minimize a convex quadratic over the Euclidean coefficient ball.

    Parameters
    ----------
    A : ndarray, shape (d, d)
        Symmetric positive-semidefinite quadratic matrix, possibly singular.
    b : ndarray, shape (d,)
        Linear coefficient; the objective is 0.5 * theta.T @ A @ theta - b.T @ theta.
    radius : float, default B
        Positive Euclidean radius.

    Returns
    -------
    theta : ndarray, shape (d,)
        Global constrained minimizer. Select the minimum-norm minimizer when
        the unconstrained optimum is nonunique and feasible.

    Raises
    ------
    ValueError
        Matrix dimensions, radius or positive-semidefiniteness are invalid.

    Notes
    -----
    Implement the KKT-based solution you derive in Markdown. Handle
    inactive/active constraints, singular designs, zero design and nonzero
    linear components in numerical null directions. Declare numerical
    tolerances. Do not substitute projection of the unconstrained optimum.
    """


    return


def fit_constrained(X_fit, y_fit, lam=0.0, reference=None):
    """
    Fit constrained least squares with one optional anchored FTRL penalty.

    Parameters
    ----------
    X_fit : ndarray, shape (n, d)
        Features on exactly the intended fitting interval.
    y_fit : ndarray, shape (n,)
        Aligned normalized outcomes.
    lam : float, default 0.0
        Nonnegative weight of one quadratic penalty on the cumulative loss.
    reference : ndarray, shape (d,), or None, default None
        Penalty center; use the zero vector when omitted.

    Returns
    -------
    theta : ndarray, shape (d,)
        Minimizer on the radius-B ball of cumulative half squared losses plus
        lam / 2 times squared distance to reference.

    Notes
    -----
    Use solve_ball_quadratic; no penalty is included when lam=0.
    The intercept is penalized when lam>0, just like all other coordinates.
    """


    return


def check_quadratic_solver(d, seed=SEED):
    """
    Independently test constrained quadratic optimality and edge cases.

    Parameters
    ----------
    d : int
        Positive feature dimension.
    seed : int, default SEED
        Seed for reproducible random numerical checks.

    Returns
    -------
    None
        Raise AssertionError if feasibility, KKT residuals or comparison with
        an independent SciPy constrained optimizer fails.

    Notes
    -----
    Include full-rank, rank-deficient, zero-design, active/inactive and
    anchored-penalty cases. Construct a counterexample to naive projection
    rather than copying an answer from a completed notebook.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## II.3 — FTL, FTRL et OGD

## Exercice 6 — Traduire les trois définitions en mises à jour de l'état

Utilisez tout l'historique révélé pour les meneurs. Soit $\theta_{\rm ref}$ l'ajustement par moindres carrés
sous contrainte sur la période de calibration. Initialisez chaque méthode principale à ce point.
Écrivez les trois mises à jour après observation et précisez l'horloge d'OGD.

Pour OGD, utilisez $\eta_k=\eta_0/\sqrt{k}$, avec $k=1$ à la première mise à jour de validation et sans réinitialisation au test.
Toutes les méthodes partent de la référence sous contrainte ajustée sur la calibration seule. Conservez les noms des classes et méthodes.
Affectez `X_cal`, `y_cal`, `theta_ref` et `FEATURE_SCALE` dans votre espace d'exécution avant les appels qui en dépendent.
Testez l'indépendance des états mutables et l'égalité des prévisions initiales.


### Votre réponse — Exercice 6

In [ ]:
def initialize_calibration(X, y, cal_slice):
    """
    Fit the common feasible calibration reference for all primary methods.

    Parameters
    ----------
    X : ndarray, shape (n, d)
        Chronological bounded features.
    y : ndarray, shape (n,)
        Aligned normalized outcomes.
    cal_slice : slice
        Unscored initial history only.

    Returns
    -------
    X_cal : ndarray, shape (N_CAL, d)
        Calibration features.
    y_cal : ndarray, shape (N_CAL,)
        Calibration outcomes.
    theta_ref : ndarray, shape (d,)
        Unregularized constrained calibration fit.

    Notes
    -----
    Bind X_cal, y_cal and theta_ref in the execution workspace before
    calling new_model. No validation or test observation may enter this fit.
    """


    return


class Leader:
    """Full-loss FTL/FTRL state with A, b, reference, lam and theta attributes."""

    def __init__(self, X_history, y_history, reference, lam=0.0):
        """
        Initialize a leader from calibration history and a fixed penalty center.

        Parameters
        ----------
        X_history : ndarray, shape (n_history, d)
            Already observed initial feature rows.
        y_history : ndarray, shape (n_history,)
            Already observed normalized outcomes.
        reference : ndarray, shape (d,)
            Calibration-only feasible reference; store an independent copy.
        lam : float, default 0.0
            Nonnegative fixed cumulative penalty; zero selects FTL.

        Returns
        -------
        None
            Create A (d, d), b (d,), reference (d,), lam (float) and theta (d,).
            theta must be the current constrained leader, not a future-data fit.

        Notes
        -----
        Store sufficient statistics rather than all historical observations.
        """


        return

    def _fit(self):
        """
        Compute the current full-loss constrained leader without adding data.

        Returns
        -------
        theta : ndarray, shape (d,)
            Minimizer using current A, b, reference and lam on the radius-B ball.

        Notes
        -----
        Do not mutate A or b and do not scale the single penalty by history length.
        """


        return

    def update(self, x, target):
        """
        Incorporate one newly revealed observation and refresh the leader.

        Parameters
        ----------
        x : ndarray, shape (d,)
            Features for the observation whose prediction has already been recorded.
        target : float
            Newly revealed normalized outcome.

        Returns
        -------
        None
            Mutate A, b and theta consistently with the full revealed history.
            Preserve the fixed reference and penalty weight.
        """


        return


class OGD:
    """Projected online-gradient state with theta, eta0 and persistent k attributes."""

    def __init__(self, initial_theta, eta0):
        """
        Initialize projected OGD without replaying calibration as scored updates.

        Parameters
        ----------
        initial_theta : ndarray, shape (d,)
            Feasible calibration coefficient vector; store an independent copy.
        eta0 : float
            Positive scale of the decreasing learning-rate schedule.

        Returns
        -------
        None
            Create theta (d,), eta0 (float), and k (int), initially zero.
        """


        return

    def update(self, x, target):
        """
        Apply one projected OGD update after the current outcome is revealed.

        Parameters
        ----------
        x : ndarray, shape (d,)
            Features for the already-recorded prediction.
        target : float
            Newly revealed normalized outcome.

        Returns
        -------
        None
            Advance k once and replace theta by the next feasible coefficient vector.

        Notes
        -----
        Use the decreasing schedule specified in Exercise 6. The first
        validation update has index 1. Keep the same clock across validation and
        test; never reset it at a plot or scoring boundary.
        """


        return


class FixedPredictor:
    """Fixed-vector baseline exposing the same theta/update protocol."""

    def __init__(self, theta):
        """
        Create a feasible fixed linear predictor.

        Parameters
        ----------
        theta : ndarray, shape (d,)
            Fixed coefficient vector; keep a private copy.

        Returns
        -------
        None
            Create theta without sharing mutable state with the caller.
        """


        return

    def update(self, x, target):
        """
        Accept feedback without changing the fixed predictor.

        Parameters
        ----------
        x : ndarray, shape (d,)
            Observed features, accepted for protocol compatibility only.
        target : float
            Observed outcome, accepted for protocol compatibility only.

        Returns
        -------
        None
            Leave theta unchanged; this method intentionally performs no learning.
        """


        return


def new_model(method, value=0.0):
    """
    Create an independent primary learner at the shared calibration state.

    Parameters
    ----------
    method : {"FTL", "FTRL", "OGD"}
        Requested primary algorithm.
    value : float, default 0.0
        FTRL penalty weight or positive OGD step scale; unused for FTL.

    Returns
    -------
    model : Leader or OGD
        Fresh object exposing theta and update(x, target), initialized using
        only the workspace globals X_cal, y_cal and theta_ref.

    Raises
    ------
    ValueError
        method is unknown or its requested hyperparameter is invalid.

    Notes
    -----
    Each call must own independent mutable state. All primary models
    must have the same first validation prediction. Do not reuse a previously
    validated or tested object when creating a new candidate.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

### Lien avec le cours — Distinguer trois pénalités quadratiques

Construisez un tableau comparant FTRL sur les pertes complètes, FTRL linéarisé (diapositives 40–43) et une perte ridge à chaque étape
(diapositives 56–59). Précisez l'objectif de la décision suivante, la perte évaluée et les hypothèses pertinentes.
Expliquez quand FTRL linéarisé et OGD coïncident, et si la projection préserve cette équivalence.
Utilisez `compare_projected_updates` sur l'exemple déclaré à deux gradients et discutez le résultat dans votre réponse à l'exercice 6.


In [ ]:
def compare_projected_updates(gradients, eta=1.0, initial_theta=0.0):
    """
    Compare projected OGD and quadratic linearized FTRL on a scalar stream.

    Parameters
    ----------
    gradients : array_like, shape (n,)
        Gradients of scalar linear convex losses; use [-2.0, 1.0] for the bridge.
    eta : float, default 1.0
        Positive constant step size.
    initial_theta : float, default 0.0
        Shared initial point in [0, 1] and quadratic penalty center.

    Returns
    -------
    comparison : pandas.DataFrame
        Columns: "After step", "Gradient", "Projected OGD", "Linearized FTRL".
        Include the post-update scalar decision for each step and method.

    Notes
    -----
    Use the feasible interval [0, 1]. This is a separate illustration, not
    the full-loss FTRL implementation used for bike demand.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## II.4 — Un moteur d'évaluation « prédire, puis apprendre »

## Exercice 7 — Enregistrer le paramètre qui a réellement produit la prévision

Implémentez une boucle réutilisable. Enregistrez les prévisions, les pertes non régularisées, les vecteurs de coefficients et
les amplitudes des mises à jour. Vérifiez que remplacer un résultat ne peut pas modifier la prévision du même jour.
Pourquoi serait-il invalide d'évaluer *après* avoir ajusté le modèle sur le résultat du jour ?

Testez la causalité par préfixe et l'identité entre FTL et FTRL à pénalité nulle.
Expliquez pourquoi il faut copier les instantanés de coefficients plutôt que conserver des alias.


### Votre réponse — Exercice 7

In [ ]:
def prequential_run(model, X_stream, y_stream):
    """
    Record causal forecasts, then update the same learner along a stream.

    Parameters
    ----------
    model : object
        Mutable learner with theta (d,) and update(x, target).
    X_stream : ndarray, shape (n, d)
        Chronological feature rows.
    y_stream : ndarray, shape (n,)
        Aligned outcomes revealed only after each corresponding prediction.

    Returns
    -------
    result : dict
        Keys: prediction (n,), loss (n,), theta (n, d), movement (n,), seconds
        (float), and model (the same learner, in its final updated state).
        theta records independent copies of pre-update coefficients; movement
        records the norm of each post-minus-pre update. loss is unregularized.

    Notes
    -----
    Predict before reading the current target. Never reset an existing
    learner clock. Validate prediction/recorded-parameter consistency and
    feasibility. Include loop/logging overhead in seconds, but no hindsight
    comparison. Calling this function mutates the passed learner.
    """


    return


def check_prequential_protocol(X_stream, y_stream):
    """
    Test prefix causality, independent states and the zero-penalty identity.

    Parameters
    ----------
    X_stream : ndarray, shape (n, d)
        Short chronological feature stream with n>=4.
    y_stream : ndarray, shape (n,)
        Aligned outcomes for causality tests.

    Returns
    -------
    None
        Raise AssertionError if current/future target perturbations affect
        already-fixed predictions, recorded coefficients are overwritten,
        or FTRL with zero penalty differs from FTL.

    Notes
    -----
    Create fresh models for each test; initialize calibration globals first.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## II.5 — Regret statique avec un véritable optimum a posteriori

## Exercice 8 — Calculer $R_n/n$ sur un intervalle évalué

Pour un intervalle évalué $\mathcal I=(t_0+1,\ldots,t_0+n)$, définissons
$$
L_n^{\rm alg}=\sum_{j=1}^n\ell_{t_0+j}(\theta_{t_0+j}),\quad
L_n^\star=\min_{u\in K}\sum_{j=1}^n\ell_{t_0+j}(u),\quad
R_n=L_n^{\rm alg}-L_n^\star.
$$
Calculez cet optimum pour chaque préfixe entièrement observé. Faut-il inclure les observations de calibration
dans le comparateur de l'intervalle évalué ? Faut-il inclure la pénalité de FTRL ?

Précisez l'intervalle et la perte définissant chaque comparateur. Distinguez un optimum fixe par préfixe entièrement observé
d'un oracle changeant chaque jour. Conservez le regret signé et les lignes de variables retardées réellement observées.


### Votre réponse — Exercice 8

In [ ]:
def prefix_oracle_losses(X_score, y_score):
    """
    Compute the best feasible fixed unregularized loss on every scored prefix.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features from the scored interval only, not its initialization history.
    y_score : ndarray, shape (n,)
        Aligned scored outcomes.

    Returns
    -------
    optimal_losses : ndarray, shape (n,)
        Entry j is the minimum cumulative loss of one fixed feasible vector
        evaluated on all j+1 observations of that completed prefix.

    Notes
    -----
    Refit the comparator separately at each horizon. Do not sum the
    last-day losses of changing oracle vectors. Use the same feasible ball,
    realized feature rows and unregularized loss as the online learners.
    Only tiny negative loss values from numerical cancellation may be removed;
    negative regret itself must never be clipped.
    """


    return


def regret_curves(result, oracle_losses):
    """
    Compare recorded cumulative learner losses with matching prefix optima.

    Parameters
    ----------
    result : dict
        prequential_run result containing a loss vector of length n.
    oracle_losses : ndarray, shape (n,)
        Exact cumulative fixed-comparator optimum for each same scored prefix.

    Returns
    -------
    regret : ndarray, shape (n,)
        Signed cumulative static regret.
    average_regret : ndarray, shape (n,)
        Signed regret divided by the corresponding positive prefix length.

    Notes
    -----
    Do not take absolute values, add algorithmic penalties or include
    calibration observations in a validation/test comparator.
    """


    return


def check_prefix_oracles(X_score, y_score, horizons=(1, 7, 50, 180)):
    """
    Validate prefix oracle accounting against direct constrained fits.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Scored features.
    y_score : ndarray, shape (n,)
        Scored outcomes.
    horizons : iterable of int, default (1, 7, 50, 180)
        Positive prefix lengths; test only values no greater than n.

    Returns
    -------
    None
        Raise AssertionError if cumulative optimum losses disagree with direct
        whole-prefix evaluations or decrease beyond numerical tolerance.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

### Travail d'implémentation — Bootstrap apparié des scores avec réajustement de l'oracle

Implémentez les fonctions documentées de comparateur par lots et pondéré. Chaque tirage bootstrap doit posséder un unique
comparateur fixe sous contrainte et non régularisé, évalué sur tout le préfixe rééchantillonné. Vérifiez l'accord des solveurs scalaire et par lots,
y compris dans les cas singuliers et ceux où la contrainte de bord est active. Enregistrez les intervalles fondés sur chaque préfixe avant de tracer.
Respectez la portée et l'historique minimal spécifiés au début du notebook.


In [ ]:
METRIC_KEYS = {
    "Total loss": "total_loss", "Mean loss": "mean_loss", "Static regret": "regret",
    "Regret / day": "average_regret", "RMSE": "rmse", "MAE": "mae", "Bias": "bias",
    "P95 abs. error": "p95_absolute_error", "Max abs. error": "max_absolute_error",
}


def batched_oracle_losses(A, b, yy, radius=B):
    """
    Solve independent unregularized constrained least-squares oracle problems.

    Parameters
    ----------
    A : ndarray, shape (n_draws, d, d)
        Positive-semidefinite weighted Gram matrices, possibly singular.
    b : ndarray, shape (n_draws, d)
        Matching weighted feature-target cross-products.
    yy : ndarray, shape (n_draws,)
        Matching weighted sums of squared targets.
    radius : float, default B
        Positive shared feasible coefficient radius.

    Returns
    -------
    losses : ndarray, shape (n_draws,)
        Minimum whole-draw half squared loss for one fixed vector per draw.
        Include yy when recovering objective values; no regularization.

    Notes
    -----
    Handle inactive and active constraints, rank deficiency and numerical
    cancellation. The scalar solve_ball_quadratic may serve as a correctness
    reference, but implement a practical batched solver for prefix intervals.
    Never approximate this oracle by clipping an unconstrained solution.
    """


    return


def weighted_oracle_losses(X_score, y_score, weights):
    """
    Fit a separate fixed comparator for each paired bootstrap weight row.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Recorded features for the available scored interval.
    y_score : ndarray, shape (n,)
        Aligned outcomes.
    weights : ndarray, shape (n_draws, n)
        Nonnegative observation multiplicities for each resampled interval.

    Returns
    -------
    losses : ndarray, shape (n_draws,)
        Minimum weighted unregularized cumulative loss on the radius-B ball.

    Notes
    -----
    Keep features and outcomes paired, including observed target lags.
    Do not recursively replace lag columns with newly generated predictions.
    """


    return


def bootstrap_prefix_metrics(X_score, y_score, predictions, phase, seed, axis_labels=None):
    """
    Estimate paired prefix-only metric intervals with refitted fixed comparators.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features for one scored interval; require n>=MIN_CI_N.
    y_score : ndarray, shape (n,)
        Matching normalized targets.
    predictions : dict[str, ndarray]
        Named recorded forecast paths, each of shape (n,).
    phase : str
        Registry prefix, such as "validation" or "test".
    seed : int
        Phase-specific reproducible seed component combined with BOOT_SEED.
    axis_labels : iterable or None, default None
        Optional n labels aligned with scored days.

    Returns
    -------
    bootstrap : dict
        Keys: names (ordered method labels), bounds (method -> metric -> (n, 2)),
        oracle_bounds (metric -> (n, 2)), final_draws (metric -> (N_BOOT, m)),
        final_oracle_draws (metric -> (N_BOOT,)), final_indices (N_BOOT, n).
        Prefix metrics: total_loss, mean_loss, regret, average_regret, rmse.
        Oracle metrics: total_loss, mean_loss, rmse. Final learner draws also
        contain mae, bias, p95_absolute_error and max_absolute_error.
        Error metrics use original rental units; losses use normalized units.

    Raises
    ------
    ValueError
        Shapes/labels are incompatible or the full interval is too short to
        supply final bootstrap draws.

    Notes
    -----
    Use fresh reproducible draws at each completed prefix, shared by all
    methods. Keep early bounds NaN before MIN_CI_N. Hold forecasts fixed and
    refit the unregularized comparator inside every draw; no retraining or
    hyperparameter reselection. Save interval arrays before any plotting.
    Use one fitted vector over the entire resampled prefix for each oracle.
    """


    return


def scorecard_intervals(point_table, bootstrap, phase):
    """
    Combine final point estimates and their paired empirical intervals.

    Parameters
    ----------
    point_table : pandas.DataFrame
        One named method per index row, with columns specified by METRIC_KEYS.
    bootstrap : dict
        Output of bootstrap_prefix_metrics for those methods and scored dates.
    phase : str
        Prefix for scorecard interval registry keys.

    Returns
    -------
    table : pandas.DataFrame
        MultiIndex (Method, Metric), with Estimate, Lower 95%, Upper 95% columns.
        Register each interval and retain the point_table values unchanged.

    Notes
    -----
    P95 is an empirical percentile; maximum-error bounds are resampling
    ranges, not valid extrapolations to unseen tail events.
    """


    return


def check_bootstrap_oracles(X_score, y_score, seed=BOOT_SEED):
    """
    Check weighted and batched comparator values against the scalar solver.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features for a small scored interval.
    y_score : ndarray, shape (n,)
        Matching targets.
    seed : int, default BOOT_SEED
        Reproducible bootstrap-check seed.

    Returns
    -------
    None
        Raise AssertionError if weighted/bootstrap losses differ from scalar
        whole-draw constrained optima. Include singular and active cases.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## II.6 — Choisir les réglages en minimisant le regret moyen de validation

## Exercice 9 — Régler les hyperparamètres sans consulter le futur

Comparez la grille de $\lambda$ spécifiée pour FTRL et la grille de $\eta_0$ pour OGD. Sélectionnez le réglage
minimisant le **$R_{180}/180$ final de validation**, et non la plus petite valeur atteinte sur un préfixe
intermédiaire arbitraire. Pourquoi ce classement est-il équivalent à celui fondé sur la perte quadratique moyenne de validation ?

Définissez l'indice de candidat $h$ et expliquez l'exposant $(h)$ dans $R_{180}^{(h)}$.
Utilisez uniquement les grilles déclarées et une règle déterministe de départage. Interprétez la sensibilité au pas et à la pénalité,
l'échelle logarithmique symétrique et la portée des bandes bootstrap ponctuelles. Distinguez la méthode sélectionnée sur la validation
d'un éventuel vainqueur ultérieur du test ; ne modifiez pas les grilles après examen des résultats du test.


### Votre réponse — Exercice 9

In [ ]:
def select_validation_settings(X_val, y_val, validation_oracle, axis_labels=None):
    """
    Search the declared grids and freeze settings using final validation average regret.

    Parameters
    ----------
    X_val : ndarray, shape (N_VAL, d)
        Validation-only chronological features.
    y_val : ndarray, shape (N_VAL,)
        Validation-only targets.
    validation_oracle : ndarray, shape (N_VAL,)
        Matching unregularized fixed-comparator prefix losses.
    axis_labels : iterable or None, default None
        Optional validation date labels for saved intervals.

    Returns
    -------
    selection : dict
        Keys: validation_runs ((method, value) -> prequential result),
        validation_table (DataFrame), selected (method -> float),
        validation_champion (Series), validation_bootstrap (dict),
        validation_curve_ci (method -> metric -> (N_VAL, 2)),
        validation_grid_ci (method -> (n_candidates, 2)),
        validation_metrics_ci (DataFrame), and figure (Matplotlib figure).
        The table contains Method, Value, Mean loss, Validation R/n,
        Validation RMSE (rentals), and Selected within method.

    Notes
    -----
    Use fresh calibration-initialized learners for FTL's single value 0.0,
    FTRL_GRID and OGD_GRID. Rank final signed R_180/180, breaking within-method
    ties by smaller Value. Identify the validation champion with deterministic
    Method/Value tie-breaking. Compute paired bootstrap arrays before plotting.
    Show searched grids, selected settings and pointwise intervals; use a
    symmetric-log regret axis. Do not access test targets or expand the grids.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

In [ ]:
def plot_validation_curves(validation_runs, selected, validation_oracle, validation_bootstrap):
    """
    Plot validation mean loss and signed average regret at the selected settings.

    Parameters
    ----------
    validation_runs : dict
        Prequential results keyed by (method, value).
    selected : dict
        Frozen within-method settings selected on final validation regret.
    validation_oracle : ndarray, shape (N_VAL,)
        Best fixed cumulative loss for every validation prefix.
    validation_bootstrap : dict
        Saved paired bootstrap output for all validation candidates.

    Returns
    -------
    figure : matplotlib.figure.Figure
        Two panels: average loss with the fixed comparator, and signed R_n/n.
        Plot saved pointwise bands with opacity CI_ALPHA and missing early bounds.

    Notes
    -----
    Include FTL, FTRL and OGD with distinct line styles. Use candidate labels
    consistent with select_validation_settings; do not retune at each prefix.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

# Partie III — Métriques finales et limites du regret par unité de temps

## III.1 — Figer les choix ; évaluer sur le flux de test chronologique

## Exercice 10 — Continuer à apprendre, mais cesser de régler les hyperparamètres

Repartez d'états de modèles neufs, rejouez la période de validation avec les réglages figés, puis évaluez
les jours de test. Vérifiez que cela produit le même état qu'une exécution ininterrompue. Ajoutez trois
prédicteurs de référence simples : l'ajustement de calibration figé, le comptage de la veille et celui de la semaine précédente.

Montrez l'équivalence avec une exécution ininterrompue de la validation puis du test. Vérifiez si les trois références fixes
sont admissibles dans la représentation choisie et distinguez les retards observés des prévisions récursives.


### Votre réponse — Exercice 10

In [ ]:
def evaluate_frozen_models(selected, validation_runs, X_val, y_val, X_test, y_test, feature_names):
    """
    Replay validation at frozen settings, then score the chronological test stream.

    Parameters
    ----------
    selected : dict
        Frozen primary-method hyperparameters.
    validation_runs : dict
        Recorded candidate runs used to verify deterministic replay.
    X_val : ndarray, shape (N_VAL, d)
        Validation features, replayed as observed history.
    y_val : ndarray, shape (N_VAL,)
        Validation outcomes.
    X_test : ndarray, shape (n_test, d)
        Later test features.
    y_test : ndarray, shape (n_test,)
        Test outcomes revealed only after their predictions.
    feature_names : list[str]
        Names aligned with feature columns, including intercept and demand lags.

    Returns
    -------
    evaluation : dict
        Keys: test_runs (six named prequential results), baseline_vectors
        (three feasible fixed coefficient arrays), and replay_seconds (method -> float).
        Baseline names: "Frozen calibration", "Previous day", "Previous week".

    Notes
    -----
    Build fresh primary states, replay validation and continue without a
    clock reset. Verify equivalence with uninterrupted validation+test runs.
    Use workspace theta_ref and FEATURE_SCALE to define fixed baselines in
    the same feature coordinates. Lag baselines use observed past counts,
    not recursively generated multi-step forecasts. Verify their feasibility.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## Exercice 11 — Calculer un tableau de résultats final et des courbes par préfixe

Ajustez le vecteur a posteriori sur les **jours de test uniquement**, puis calculez le regret total, le regret par jour de test,
la RMSE, la MAE, le biais signé et une métrique d'erreur extrême. Pourquoi l'oracle doit-il être non régularisé,
même lorsque l'apprenant est FTRL ?

Précisez toutes les unités et la convention de signe du biais. Vérifiez la conversion entre perte normalisée et RMSE.
Associez au tableau des intervalles appariés avec réajustement du comparateur et tracez les courbes par préfixe de test.
Comparez OGD aux références quotidienne et hebdomadaire avec des tirages appariés de réduction de RMSE. Rédigez votre
interprétation finale à partir des résultats effectivement obtenus ; ne supposez pas qu'une méthode doive gagner.


### Votre réponse — Exercice 11

In [ ]:
def evaluate_test_metrics(X_test, y_test, test_runs, baseline_vectors, axis_labels=None):
    """
    Compute the final independent test scorecard and matching empirical intervals.

    Parameters
    ----------
    X_test : ndarray, shape (n_test, d)
        Test-only features.
    y_test : ndarray, shape (n_test,)
        Test-only normalized outcomes.
    test_runs : dict
        Six causal prequential results; add regret and average_regret arrays
        to each run only after predictions have been completed.
    baseline_vectors : dict[str, ndarray]
        Three fixed feasible baseline coefficient vectors.
    axis_labels : iterable or None, default None
        Optional test date labels for saved interval bounds.

    Returns
    -------
    scores : dict
        Keys: metrics (DataFrame indexed by Method), metrics_ci (DataFrame),
        test_oracle_losses (n_test,), theta_test_oracle (d,), oracle_direct_loss
        (float), oracle_seconds (float), test_bootstrap (dict), test_curve_ci (dict).
        metrics contains Days and every display column specified by METRIC_KEYS.

    Notes
    -----
    Fit the comparator only on test dates with no penalty. Verify the full
    fit agrees with the final prefix optimum and does not lose to any feasible
    fixed baseline. Preserve signed adaptive regret. RMSE, MAE, bias, P95 and
    maximum errors are in rental units; bias is prediction minus observation.
    Refit comparators in paired bootstrap draws and save all bounds before plots.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

In [ ]:
def plot_test_curves(test_runs, test_oracle_losses, test_bootstrap):
    """
    Plot test cumulative loss, static regret and regret per scored day.

    Parameters
    ----------
    test_runs : dict
        Results including regret and average_regret for FTL, FTRL and OGD.
    test_oracle_losses : ndarray, shape (n_test,)
        Cumulative fixed-comparator optima for test prefixes.
    test_bootstrap : dict
        Previously saved learner and oracle prefix interval arrays.

    Returns
    -------
    figure : matplotlib.figure.Figure
        Three panels with aligned test-prefix axes and pointwise interval bands.
        Include the prefix oracle in the cumulative-loss panel and preserve
        negative regrets in the other panels.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

In [ ]:
def compare_test_models(metrics, metrics_ci, test_bootstrap, selected, validation_champion):
    """
    Summarize the observed test comparison without changing the selection rule.

    Parameters
    ----------
    metrics : pandas.DataFrame
        Final six-model test scorecard.
    metrics_ci : pandas.DataFrame
        Saved final metric intervals indexed by (Method, Metric).
    test_bootstrap : dict
        Paired final bootstrap draws aligned across all models.
    selected : dict
        Frozen validation-selected settings.
    validation_champion : pandas.Series
        Primary model chosen on validation, not chosen retrospectively on test.

    Returns
    -------
    comparison : dict
        Keys: summary (DataFrame of observed comparisons) and rmse_gain_ci
        (daily/weekly baseline name -> two-element interval of percentage RMSE
        reduction for OGD). Register the paired comparison bounds.

    Notes
    -----
    Compute conclusions from completed results, not from a preferred winner.
    Distinguish the validation-selected method from the best realized test
    method. Use paired draws for percentage RMSE comparisons and explain
    selection/conditional-interval limits in your Markdown answer.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## III.2 — Erreurs au fil du temps, comportement extrême et coût de calcul

## Exercice 12 — Aller au-delà de la moyenne globale

Tracez la trajectoire complète avec les frontières des périodes, les 120 dernières prévisions de test, la RMSE glissante
et les plus grandes erreurs d'OGD. Examinez les normes des coefficients et les amplitudes des mises à jour. Une baisse
du regret moyen implique-t-elle une amélioration de chaque prévision récente ?

Ajoutez les normes moyennes des coefficients et les amplitudes moyennes des mises à jour sur 28 jours, avec des bandes bootstrap enregistrées et des trajectoires brutes en transparence.
Répétez des exécutions indépendantes pour estimer l'incertitude des temps de calcul séparément du rééchantillonnage calendaire. Expliquez la mémoire de l'état en ligne,
le surcoût d'enregistrement, le travail par mise à jour et le coût distinct de l'évaluation a posteriori.


### Votre réponse — Exercice 12

In [ ]:
def plot_forecast_diagnostics(daily, dates, cal_slice, val_slice, test_slice, y_test, test_runs):
    """
    Inspect realized forecasts, recent errors and large test mistakes after selection.

    Parameters
    ----------
    daily : pandas.DataFrame
        Raw observations; all-period inspection is allowed only now.
    dates : pandas.DatetimeIndex
        Usable feature dates.
    cal_slice : slice
        Initial calibration interval.
    val_slice : slice
        Validation interval.
    test_slice : slice
        Final test interval.
    y_test : ndarray, shape (n_test,)
        Normalized test outcomes.
    test_runs : dict
        Recorded causal forecasts for primary methods and fixed baselines.

    Returns
    -------
    diagnostics : dict
        Keys: figure (three-panel Figure), worst_errors (top-eight OGD error
        DataFrame), rolling_rmse (n_test, 4) and rolling_rmse_ci (n_test, 4, 2).
        Rolling columns: FTL, FTRL, OGD, Previous day. Register interval arrays.

    Notes
    -----
    Show the whole trajectory with split shading, the final 120 forecasts,
    and 30-day trailing RMSE with paired bootstrap bands. Add no confidence
    band to a single recorded forecast/outcome. Do not feed diagnostics back
    into hyperparameter selection or infer disruption causes from counts alone.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

In [ ]:
def diagnose_stability_and_cost(test_runs, selected, X_val, y_val, X_test, y_test, test_dates):
    """
    Measure coefficient stability and repeated fresh-run computational cost.

    Parameters
    ----------
    test_runs : dict
        Recorded primary-model coefficient and movement paths.
    selected : dict
        Frozen settings for FTL, FTRL and OGD.
    X_val : ndarray, shape (N_VAL, d)
        Validation features for fresh timing replays.
    y_val : ndarray, shape (N_VAL,)
        Validation outcomes.
    X_test : ndarray, shape (n_test, d)
        Test features.
    y_test : ndarray, shape (n_test,)
        Test outcomes.
    test_dates : pandas.DatetimeIndex
        Dates aligned with recorded state paths.

    Returns
    -------
    diagnostics : dict
        Keys: figure (norm/movement Figure), cost_table (DataFrame),
        timing_samples (N_TIMING_REPEATS, 3, 2), oracle_seconds (float),
        oracle_time_ci (2,), norm_mean_ci (n_test, 3, 2), and step_mean_ci
        (n_test, 3, 2). Save the empirical interval arrays.

    Notes
    -----
    Show 28-day trailing means with paired block bands and faint raw state
    traces. Use repeated fresh learners for timing, with separate replay/test
    means and bootstrap intervals over timing repetitions, not calendar days.
    Measure common hindsight evaluation separately from online update costs.
    Explain state memory, logging overhead and dimension-dependent work.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

### RMSE agrégée sur l'ensemble de la période de prévision, avec intervalles empiriques à 95 %

Ce diagramme en barres demandé comprend **les 634 jours de prévision** : validation et test,
hors calibration. Calculez `sqrt(mean(squared error))` sur toute cette période ;
ne faites **pas** la moyenne des RMSE glissantes ni des erreurs absolues quotidiennes. Les six modèles utilisent
les mêmes dates et les mêmes tirages de blocs appariés. Puisque la validation a servi à choisir les réglages, ce graphique
est descriptif ; le tableau de résultats indépendant sur le test chronologique reste celui de l'exercice 11.


### Votre interprétation

In [ ]:
def pooled_forecast_rmse(validation_runs, selected, test_runs, baseline_vectors, X_val, y_val, y_test, axis_labels=None):
    """
    Compare pooled RMSE over validation and test, excluding calibration.

    Parameters
    ----------
    validation_runs : dict
        Recorded validation candidate runs.
    selected : dict
        Frozen settings identifying each primary validation path.
    test_runs : dict
        Six recorded test forecast paths.
    baseline_vectors : dict[str, ndarray]
        Fixed feasible reference vectors used on both intervals.
    X_val : ndarray, shape (N_VAL, d)
        Validation features for fixed-baseline predictions.
    y_val : ndarray, shape (N_VAL,)
        Validation targets.
    y_test : ndarray, shape (n_test,)
        Test targets.
    axis_labels : iterable or None, default None
        Optional six labels for registered model-level RMSE bounds.

    Returns
    -------
    pooled : dict
        Keys: forecast_rmse_summary (six-row DataFrame), forecast_rmse_ci
        (6, 2), forecast_predictions (method -> 634 normalized forecasts),
        and figure (bar chart with empirical interval endpoints).

    Notes
    -----
    Pool all 634 forecast errors before taking RMSE; do not average rolling
    RMSEs or one-day absolute errors. Pair block draws across models and plot
    all interval endpoints, which need not be symmetric around point estimates.
    This is descriptive because validation participated in selection; it does
    not replace the independent final-test scorecard.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## III.3 — Vérifier une borne d'OGD en temps fini sans exagérer sa portée

## Exercice 13 — Relier l'implémentation à la garantie du cours

Utilisez $\|\theta\|\le B=4$, $\|x_t\|\le1$, $|y_t|\le1$.
Déterminez $D$ et $G$. Le cours établit une borne à pas constant. Qu'est-ce qui change pour notre suite de pas
décroissants, et peut-on déduire une vitesse logarithmique du caractère quadratique de la perte ?

Établissez une borne valide avec le décalage de validation dans l'horloge du test, puis comparez ses formes fondées sur une borne uniforme du gradient et sur
les gradients réalisés au regret observé sur le test. Énoncez précisément l'implication asymptotique unilatérale de la propriété sans regret,
et discutez le regret négatif, la monotonie et ce qu'une courbe finie permet d'établir. N'attribuez pas d'intervalles de confiance statistiques
à des inégalités analytiques exactes.


### Votre réponse — Exercice 13

In [ ]:
def check_ogd_bound(X_test, y_test, ogd_run, eta0, offset=N_VAL):
    """
    Evaluate the finite-horizon OGD bounds derived in your written answer.

    Parameters
    ----------
    X_test : ndarray, shape (n_test, d)
        Test features satisfying the stated norm assumption.
    y_test : ndarray, shape (n_test,)
        Test outcomes satisfying the stated outcome assumption.
    ogd_run : dict
        Recorded OGD result including pre-update theta, predictions and loss.
    eta0 : float
        Positive validation-selected step scale.
    offset : int, default N_VAL
        Number of scored updates already completed before test; do not reset it.

    Returns
    -------
    bound_table : pandas.DataFrame
        Rows for observed static regret, the bound using observed gradients and
        the bound using a uniform gradient bound. Columns: Cumulative and
        Per test day. Exact theoretical bounds have no sampling intervals.

    Notes
    -----
    Derive D, G and the nonconstant-step inequality yourself before coding.
    Verify the step schedule is positive/nonincreasing, gradients respect the
    uniform bound, and empirical regret is no larger than the derived bounds.
    Use the post-validation feasible initial state and a test-only comparator.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## III.4 — Pourquoi le regret moyen statique ne suffit pas

## Exercice 14 — Construire un prédicteur sans regret mais imprécis

Soient $K=[0,1]$, $\widehat y_t=1/2$ fixé avant toute observation et un flux constitué de
200 zéros suivis de 200 uns. Calculez son regret moyen statique et sa perte quadratique moyenne.
Comparez-le à une référence a posteriori autorisée à utiliser un coefficient par bloc de 200 jours.

Évaluez aussi le même prédicteur fixé à l'avance sur un flux de zéros de même longueur. Expliquez ce que cet exemple
construit établit ou non concernant les garanties universelles et le modèle réel à 13 variables.


### Votre réponse — Exercice 14

In [ ]:
def analyze_toy_stream(block_length=200, prediction=0.5):
    """
    Evaluate the fixed scalar predictor on a two-regime stream and a second stream.

    Parameters
    ----------
    block_length : int, default 200
        Positive number of zeros followed by the same number of ones.
    prediction : float, default 0.5
        Fixed prediction declared before any outcomes; feasible set is [0, 1].

    Returns
    -------
    results : pandas.Series
        Static regret per day, mean half squared loss, MAE, two-block regret
        per day, and the same predictor's regret per day on an all-zero stream.
    figure : matplotlib.figure.Figure
        Deterministic outcomes and the fixed prediction, with no sampling bands.

    Notes
    -----
    Fit one fixed scalar hindsight comparator for the full balanced stream
    and one per known block for the stronger reference. Evaluate the same
    predeclared predictor on the all-zero stream as a separate check. Compute
    all values rather than hard-coding them; interpret them in Markdown.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## Exercice 15 — Renforcer la référence sur les données réelles

Sur l'intervalle de test réel, répartissez les jours en blocs consécutifs fixes de 13 semaines (91 jours), le dernier
bloc pouvant être plus court. Ajustez un vecteur de coefficients dans $K$ par bloc. Comparez l'excès moyen
de perte face à cet oracle plus riche au regret moyen statique. Le théorème initial couvre-t-il ce cas ?

Gardez les frontières fixes. Pour les intervalles empiriques, rééchantillonnez à l'intérieur des blocs prédéfinis et réajustez
les comparateurs statique et par blocs sur le même tirage. Expliquez ce que la souplesse supplémentaire permet ou non de conclure
sur la dérive des coefficients, et donnez une contrainte possible sur la variation du comparateur pour une garantie dynamique.


### Votre réponse — Exercice 15

In [ ]:
def blockwise_tracking(X_test, y_test, test_runs, test_dates, block_size=91):
    """
    Compare static regret with excess loss against a predetermined blockwise oracle.

    Parameters
    ----------
    X_test : ndarray, shape (n_test, d)
        Test-only observed feature rows.
    y_test : ndarray, shape (n_test,)
        Test-only normalized targets.
    test_runs : dict
        Completed primary-model forecast/loss paths.
    test_dates : pandas.DatetimeIndex
        Aligned chronological test dates.
    block_size : int, default 91
        Predetermined consecutive block length, with the last block shorter if needed.

    Returns
    -------
    tracking : dict
        Keys: block_table (dates, sizes, oracle mean losses), tracking
        (method-indexed static and blockwise regret/day DataFrame),
        block_ci (DataFrame), and tracking_metrics_ci (DataFrame).
        Register block and model comparison intervals.

    Notes
    -----
    Fit one feasible unregularized fixed vector per block. Verify blockwise
    oracle loss never exceeds global static oracle loss. For intervals, keep
    block boundaries fixed and resample within each block, pairing all methods
    and refitting both comparators on the same combined draw. Do not optimize
    boundaries using targets or treat this hindsight oracle as deployable.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## III.5 — Autres limites : la perte, l'information et le contexte de déploiement

## Exercice 16 — Et si sous-prévoir coûtait trois fois plus cher ?

Supposons qu'une décision hypothétique sur une période, $q\in[0,Q_{\max}]$, ait le coût
$$
C(q,y)=c_h(q-y)_++c_b(y-q)_+,\qquad c_h=1,\ c_b=3.
$$
Quelle fonctionnelle conditionnelle faut-il prévoir ? Déterminez un sous-gradient valide en $q$.
Notre regret pour la perte quadratique certifierait-il une bonne performance pour ce coût ?

Incluez le cas d'égalité du sous-gradient et les contraintes de faisabilité pour une décision linéaire.
Expliquez le comparateur approprié et si les comptages de locations observés permettent une interprétation littérale en gestion des stocks.
La cellule Python ci-dessous accompagne votre démonstration ; elle ne définit pas un nouveau modèle de prévision des locations.


### Votre réponse — Exercice 16

In [ ]:
def inventory_cost(q, target, holding_cost=1.0, shortage_cost=3.0):
    """
    Evaluate the asymmetric one-period surplus/shortage loss from Exercise 16.

    Parameters
    ----------
    q : float
        Feasible nonnegative decision quantity.
    target : float
        Observed demand in the same units.
    holding_cost : float, default 1.0
        Positive cost per unit of surplus.
    shortage_cost : float, default 3.0
        Positive cost per unit of unmet demand.

    Returns
    -------
    cost : float
        The stated piecewise-linear inventory loss, not a squared error.
    """


    return


def inventory_subgradient(q, target, holding_cost=1.0, shortage_cost=3.0):
    """
    Choose a valid scalar subgradient of inventory_cost with respect to q.

    Parameters
    ----------
    q : float
        Current decision quantity.
    target : float
        Revealed demand.
    holding_cost : float, default 1.0
        Positive surplus unit cost.
    shortage_cost : float, default 3.0
        Positive shortage unit cost.

    Returns
    -------
    subgradient : float
        A valid derivative away from equality and a valid selected subgradient
        at equality; state your equality convention in the written answer.
    """


    return


def projected_inventory_update(q, target, eta, capacity, holding_cost=1.0, shortage_cost=3.0):
    """
    Perform one feedback-based inventory update within the feasible quantity interval.

    Parameters
    ----------
    q : float
        Current quantity in [0, capacity].
    target : float
        Newly observed demand.
    eta : float
        Positive update step.
    capacity : float
        Positive upper bound on feasible quantity.
    holding_cost : float, default 1.0
        Surplus unit cost.
    shortage_cost : float, default 3.0
        Shortage unit cost.

    Returns
    -------
    q_next : float
        Updated quantity in [0, capacity], using your selected subgradient.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## Exercice 17 — Expliquer la distinction avec ridge

Si nous définissons $\widetilde\ell_t(\theta)=\ell_t(\theta)+\gamma\|\theta\|^2/2$, $\gamma>0$,
qu'est-ce qui change dans le gradient, le comparateur et le théorème de regret ? Est-ce la même chose que notre FTRL ?

Tenez compte de la constante, de la longueur de l'intervalle évalué et des hypothèses sur les pas d'apprentissage.
Expliquez la différence entre une pénalité incluse dans chaque perte évaluée et une pénalité algorithmique fixe
sur les pertes cumulées non régularisées. Implémentez les fonctions ridge documentées séparément de l'expérience principale.


### Votre réponse — Exercice 17

In [ ]:
def ridge_loss(theta, x, target, gamma):
    """
    Evaluate the per-observation ridge-regularized loss defined in Exercise 17.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Coefficients, including the intercept coordinate.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized observed outcome.
    gamma : float
        Positive penalty weight applied to every observation and coordinate.

    Returns
    -------
    loss : float
        Regularized per-round loss; unlike the main experiment, evaluation
        includes the per-round penalty.
    """


    return


def ridge_gradient(theta, x, target, gamma):
    """
    Evaluate the gradient of the per-round ridge loss.

    Parameters
    ----------
    theta : ndarray, shape (d,)
        Coefficients, including the intercept.
    x : ndarray, shape (d,)
        Scaled feature vector.
    target : float
        Normalized target.
    gamma : float
        Positive per-round penalty weight.

    Returns
    -------
    gradient : ndarray, shape (d,)
        Gradient in the same scaled coordinate system as theta.
    """


    return


def fit_ridge_comparator(X_score, y_score, gamma):
    """
    Fit the fixed feasible comparator for a sum of per-round ridge losses.

    Parameters
    ----------
    X_score : ndarray, shape (n, d)
        Features from exactly the scored interval.
    y_score : ndarray, shape (n,)
        Aligned normalized outcomes.
    gamma : float
        Positive penalty weight appearing in every scored loss.

    Returns
    -------
    theta : ndarray, shape (d,)
        Feasible minimizer of the summed ridge_loss objectives. Account for
        interval length when assembling the cumulative penalty.

    Notes
    -----
    This comparator belongs to a different evaluated loss from the main
    FTL/FTRL/OGD experiment; explain the distinction before implementing it.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## Intervalles empiriques enregistrés — examiner ou réutiliser les bornes exactes des graphiques

Construisez les tableaux **avant** les graphiques correspondants. Complétez la fonction ci-dessous pour exporter chaque
intervalle avec son indice ou libellé d'axe associé, ainsi que les tableaux de résultats :

- `validation_grid_ci[method]` : une paire `[lower_95, upper_95]` par hyperparamètre essayé ;
- `validation_curve_ci[method]["mean_loss" or "average_regret"]` : une ligne par préfixe de validation ;
- `test_curve_ci[method]["total_loss", "regret", "average_regret", ...]` : une ligne par préfixe de test ;
- `metrics_ci` : chaque statistique de performance finale sur le test, pour les six modèles ;
- `forecast_rmse_ci` : une ligne par modèle pour le diagramme en barres couvrant toute la période de prévision ;
- `CI_BOUNDS` : un registre contenant aussi les intervalles de l'analyse exploratoire, des fenêtres locales, des temps de calcul et du suivi.

Par exemple, `test_curve_ci["OGD"]["average_regret"][n - 1]` est l'intervalle enregistré
après exactement `n` prévisions de test. Les bornes valent `NaN` tant que quatre blocs hebdomadaires
ne sont pas disponibles ; aucune donnée future n'est empruntée pour créer une bande étroite à une date précoce.
Les fichiers NPZ se chargent avec `allow_pickle=False` ; le CSV fournit une vue indexée lisible.


### Votre interprétation

In [ ]:
def export_empirical_intervals(results_dir, tables, manifest):
    """
    Validate and export student-computed interval arrays without overwriting instructor results.

    Parameters
    ----------
    results_dir : pathlib.Path
        Use a dedicated student directory, for example
        online_learning_bike_sharing_student_results/en or its fr counterpart.
    tables : dict[str, pandas.DataFrame]
        Named final/validation/pooled/tracking/timing scorecards to export as CSV.
    manifest : dict
        Record CI_LEVEL, N_BOOT, BLOCK_LENGTH, BOOT_SEED, MIN_CI_N, CI_ALPHA,
        fixed-forecast/refitted-oracle semantics, pointwise limitations and timing
        resampling details. State the descriptive status of pooled forecast RMSE.

    Returns
    -------
    files : dict[str, pathlib.Path]
        Include empirical_95_ci_bounds.npz, empirical_95_ci_axes.npz,
        empirical_95_ci_bounds.csv, empirical_95_ci_manifest.json and one CSV
        per supplied scorecard. No object arrays or pickle.

    Notes
    -----
    Read CI_BOUNDS and CI_AXES. Validate shape, endpoint order, early NaNs,
    matching labels and prefix loss/regret scaling. The registries contain
    endpoints only: perform draw-level comparator checks upstream in the
    oracle and prequential verification functions, not in this exporter.
    Build and check all arrays before export. Reload NPZ with allow_pickle=False
    and verify equality. Do not load instructor result files as exercise answers.
    Keep English and French student exports in separate directories.
    """


    return


**Espace d’exécution :** appelez vos fonctions complétées, affectez leurs sorties et lancez vos contrôles ici.

## Synthèse finale — Rédiger vos propres conclusions

Complétez un tableau à trois colonnes : **résultat observé**, **ce qu'il permet d'établir** et **ce qu'il ne démontre pas**.
Traitez le faible regret moyen sur le test, une borne supérieure sans regret, un comparateur fixe, la sélection sur grille,
les pertes convexes bornées, la performance quadratique et une courbe finie favorable.
Discutez les limites de ce flux de deux ans dans une seule ville, la disponibilité du retour quotidien, la censure de la demande,
les conditions changeantes, les autres méthodes de suivi et les réglages répétés sur le test.
Ne recopiez pas une conclusion complétée : utilisez vos résultats exécutés et deux limites distinctes pour le déploiement.

### Liste de vérification

- [ ] Jeu de données ouvert avec attribution, README original et cache local vérifié.
- [ ] Analyse descriptive limitée à la calibration et variables strictement causales.
- [ ] Perte convexe commune, ensemble admissible borné et protocole de retour d'information explicite.
- [ ] FTL, FTRL sur les pertes complètes et OGD à implémenter et à vérifier numériquement.
- [ ] Référence exacte a posteriori sous contrainte pour chaque préfixe, sans raccourci à oracle changeant.
- [ ] Sélection des réglages figés par le regret de validation minimal par jour ; le test reste séquentiel.
- [ ] Diagnostics finaux de précision, de regret, d'erreurs extrêmes, de stabilité et de coût.
- [ ] Intervalles empiriques appariés ponctuels à 95 %, bornes enregistrées et graphique de RMSE sur toute la période de prévision.
- [ ] Borne en temps fini, contre-exemple avec une référence faible et diagnostic de suivi plus exigeant.
- [ ] Réponses aux distinctions concernant ridge et la perte asymétrique.

### Compte rendu étudiant final (une page)

Énoncez la perte et l'ensemble d'information ; décrivez le découpage et la mise à l'échelle ; présentez les réglages
sélectionnés sur la validation et le tableau de résultats final ; interprétez un graphique d'erreurs ; expliquez **deux raisons distinctes**
pour lesquelles un faible regret moyen statique ne suffit pas pour le déploiement. Appuyez-vous sur les tableaux exécutés plutôt que
d'affirmer qu'un algorithme préféré doit nécessairement gagner.

### Références et reproductibilité

1. Paul Minchella, *Apprentissage en ligne*, [diapositives du cours](../Cours_Online_Learning.pdf), en particulier les diapositives
   imprimées 26–29, 32–49 et 53–65.
2. Hadi Fanaee-T, *Vélos en libre-service*, dépôt UCI d'apprentissage automatique,
   [doi:10.24432/C5W894](https://doi.org/10.24432/C5W894),
   [page du jeu de données](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), CC BY 4.0.
3. Fanaee-T et Gama (2013),
   [doi:10.1007/s13748-013-0040-3](https://doi.org/10.1007/s13748-013-0040-3).
4. Elad Hazan, *Introduction à l'optimisation convexe en ligne*, pour les démonstrations de regret et leurs extensions.

La graine ne contrôle que les vérifications numériques, pas les observations. Aucun découpage avec mélange aléatoire n'est utilisé.
Avec des versions de bibliothèques fixées, les données, les réglages et les prévisions sont déterministes à la précision numérique
près ; les mesures de temps d'exécution ne le sont pas. Consignez les versions de vos bibliothèques dans la partie 0.
**Redémarrez le noyau et exécutez toutes les cellules** après avoir complété votre implémentation. Conservez le notebook et
le répertoire `online_learning_bike_sharing_data` ensemble pour une réexécution hors ligne.

**Reproductibilité du bootstrap.** La graine de rééchantillonnage, la longueur des blocs, le nombre de répétitions, l'historique
minimal et les réserves concernant la couverture ponctuelle sont consignés dans
`online_learning_bike_sharing_student_results/fr/empirical_95_ci_manifest.json`. Les estimations ponctuelles
ne sont pas modifiées par l'ajout des intervalles. Les bandes empiriques ne sont pas des séquences de confiance et ne
garantissent pas une couverture simultanée à 95 %, ni une couverture sans hypothèse de distribution en présence de dérive.


### Votre interprétation